# Business Email and Report Manager
### Complete AI-Powered Business Assistant
**Agentic AI Bootcamp - atomcamp | Weekly Project Assignment**

**Features:**
1. Smart Email Writer
2. Report Generator
3. Meeting Summarizer
4. Data Analyzer
5. Client Communication Drafter

**Tools:**
1. Calculator
2. Web Search (mock)
3. Data Analyzer
4. Report Formatter

> Uses **Groq API** (free). Add your key to Colab Secrets as `groq`.

---
## Step 1: Install Dependencies

In [ ]:
!pip install groq --quiet
print("Dependencies installed.")

---
## Step 2: Load API Key from Colab Secrets

In [ ]:
import os

# Load from Colab Secrets (secret name: 'groq')
try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('groq')
    if GROQ_API_KEY:
        print("Groq API key loaded from Colab Secrets.")
    else:
        raise ValueError("Key is empty.")
except Exception:
    GROQ_API_KEY = os.environ.get("GROQ_API_KEY", "")
    if GROQ_API_KEY:
        print("Groq API key loaded from environment variable.")
    else:
        print("API key not found. Add 'groq' to Colab Secrets (lock icon in the left sidebar).")

os.environ["GROQ_API_KEY"] = GROQ_API_KEY

---
## Step 3: Imports and Global Setup

In [ ]:
import json
import math
import statistics
from datetime import datetime
from groq import Groq

print("Imports complete.")

---
## Step 4: Define All Tool Functions

In [ ]:
# ============================================================
# TOOL 1: Calculator
# Handles financial math, percentages, and growth rates
# ============================================================

def calculate(expression: str) -> str:
    """
    Safely evaluates a mathematical expression.
    Supports: +, -, *, /, **, sqrt, log, abs, round, pi, e
    Examples: '50000 * 0.15', '(75000 - 50000) / 50000 * 100', 'sqrt(144)'
    """
    try:
        if not expression or not expression.strip():
            return json.dumps({"error": "Expression cannot be empty.", "status": "error"})

        safe_context = {
            "__builtins__": {},
            "sqrt": math.sqrt,
            "log": math.log,
            "log10": math.log10,
            "sin": math.sin,
            "cos": math.cos,
            "abs": abs,
            "round": round,
            "pi": math.pi,
            "e": math.e,
        }
        result = eval(expression, safe_context)
        return json.dumps({
            "expression": expression,
            "result": round(float(result), 4),
            "status": "success"
        })
    except ZeroDivisionError:
        return json.dumps({"error": "Division by zero.", "status": "error"})
    except Exception as e:
        return json.dumps({"error": f"Calculation failed: {str(e)}", "status": "error"})


# ============================================================
# TOOL 2: Web Search (Mock)
# Returns realistic business and market data
# ============================================================

def web_search(query: str) -> str:
    """
    Mock web search for business research.
    Covers: market trends, industry news, best practices, competitors, finance.
    Replace with SerpAPI or Tavily for production use.
    """
    if not query or not query.strip():
        return json.dumps({"error": "Search query cannot be empty.", "status": "error"})

    q = query.lower()

    mock_data = {
        "market trend": [
            {"title": "Global Tech Market Q1 2026", "snippet": "The technology sector recorded 15% YoY growth in Q1 2026, driven by AI adoption and cloud migration. SaaS revenue grew by 22% globally."},
            {"title": "Emerging Market Opportunities", "snippet": "Southeast Asia and South Asia are seeing the fastest B2B software adoption rates, with Pakistan's IT exports surpassing $3 billion in FY2025."}
        ],
        "industry news": [
            {"title": "AI Adoption in Business 2026", "snippet": "Over 67% of mid-size enterprises have integrated AI tools into daily operations as of early 2026, up from 34% in 2024."},
            {"title": "Digital Transformation Report", "snippet": "Companies investing in digital transformation report 30% higher customer retention and 20% lower operational costs on average."}
        ],
        "email best practice": [
            {"title": "Professional Email Guide 2026", "snippet": "High-performing business emails have clear subject lines under 50 characters, a single call-to-action, and are sent on Tuesday-Thursday mornings."},
            {"title": "B2B Email Benchmarks", "snippet": "Average B2B email open rate is 22.5%. Personalized subject lines increase open rates by 26%. Follow-up emails generate 16% more responses."}
        ],
        "competitor": [
            {"title": "Competitive Landscape Analysis", "snippet": "Major competitors are expanding into adjacent markets. Three top players have raised Series B/C funding in Q1 2026, signaling aggressive growth plans."},
            {"title": "Market Share Report", "snippet": "The top 5 players control 58% of the addressable market. Smaller entrants are competing on pricing and niche specialization."}
        ],
        "sales strategy": [
            {"title": "Sales Effectiveness Report 2026", "snippet": "Consultative selling outperforms transactional approaches by 43%. Account-based sales strategies see 28% higher deal values."},
            {"title": "Revenue Growth Tactics", "snippet": "Top-performing sales teams invest heavily in post-sale customer success, achieving 35% revenue from upsells and renewals."}
        ],
        "finance": [
            {"title": "Business Finance Trends Q1 2026", "snippet": "SME lending rates stabilized at 8-11% range. Invoice financing and revenue-based financing gaining popularity among startups."},
            {"title": "Cash Flow Management Best Practices", "snippet": "Businesses maintaining 3-month operating reserves report 40% lower risk of operational disruptions during market downturns."}
        ],
        "marketing": [
            {"title": "Digital Marketing ROI 2026", "snippet": "Content marketing delivers 3x more leads than outbound at 62% lower cost. Video content has 4x higher engagement than static posts."},
            {"title": "B2B Marketing Channels", "snippet": "LinkedIn remains the top B2B channel with 80% of leads. Email marketing ROI averages $36 for every $1 spent in 2025-2026."}
        ]
    }

    results = []
    for keyword, data in mock_data.items():
        if keyword in q:
            results = data
            break

    if not results:
        results = [
            {
                "title": f"Business Research: {query.title()}",
                "snippet": f"Recent analysis on '{query}' indicates growing market activity. Industry experts recommend staying updated with quarterly reports and aligning strategy to current demand signals."
            },
            {
                "title": f"{query.title()} - Industry Insights 2026",
                "snippet": f"Organizations focused on '{query}' are seeing increased ROI by adopting data-driven approaches and automation. Cross-functional alignment is cited as a key success factor."
            }
        ]

    return json.dumps({
        "query": query,
        "result_count": len(results),
        "results": results,
        "status": "success"
    })


# ============================================================
# TOOL 3: Data Analyzer
# Analyzes sales, revenue, and business metric datasets
# ============================================================

def analyze_data(data_string: str, operation: str = "all") -> str:
    """
    Analyzes business data provided as a JSON string.
    Accepts list format: [50000, 65000, 70000]
    Accepts dict format: {"Jan": 50000, "Feb": 65000, "Mar": 70000}
    Operations: sum, average, max, min, median, std, count, all
    """
    try:
        if not data_string or not data_string.strip():
            return json.dumps({"error": "Data string cannot be empty.", "status": "error"})

        raw = json.loads(data_string)

        # Support both list and dict formats
        if isinstance(raw, list):
            values = [float(x) for x in raw]
            labels = [str(i + 1) for i in range(len(values))]
        elif isinstance(raw, dict):
            labels = list(raw.keys())
            values = [float(v) for v in raw.values()]
        else:
            return json.dumps({"error": "Data must be a JSON list or object.", "status": "error"})

        if not values:
            return json.dumps({"error": "Dataset is empty.", "status": "error"})

        op = operation.lower().strip()

        if op == "all":
            max_val = max(values)
            min_val = min(values)
            max_label = labels[values.index(max_val)]
            min_label = labels[values.index(min_val)]

            # Trend: compare first half vs second half
            mid = len(values) // 2
            if mid > 0:
                first_half_avg = sum(values[:mid]) / mid
                second_half_avg = sum(values[mid:]) / (len(values) - mid)
                if second_half_avg > first_half_avg * 1.02:
                    trend = "Increasing"
                elif second_half_avg < first_half_avg * 0.98:
                    trend = "Decreasing"
                else:
                    trend = "Stable"
            else:
                trend = "Insufficient data for trend"

            # Period-over-period growth (first to last)
            growth_rate = None
            if len(values) >= 2 and values[0] != 0:
                growth_rate = round((values[-1] - values[0]) / values[0] * 100, 2)

            result = {
                "count": len(values),
                "labels": labels,
                "sum": round(sum(values), 2),
                "average": round(statistics.mean(values), 2),
                "median": round(statistics.median(values), 2),
                "max": {"value": max_val, "label": max_label},
                "min": {"value": min_val, "label": min_label},
                "std_dev": round(statistics.stdev(values), 2) if len(values) > 1 else 0,
                "trend": trend,
                "growth_rate_percent": growth_rate
            }
        elif op in ("sum", "total"):
            result = {"sum": round(sum(values), 2)}
        elif op in ("average", "mean", "avg"):
            result = {"average": round(statistics.mean(values), 2)}
        elif op == "max":
            m = max(values)
            result = {"max": m, "label": labels[values.index(m)]}
        elif op == "min":
            m = min(values)
            result = {"min": m, "label": labels[values.index(m)]}
        elif op == "median":
            result = {"median": round(statistics.median(values), 2)}
        elif op in ("std", "stdev", "std_dev"):
            result = {"std_dev": round(statistics.stdev(values), 2) if len(values) > 1 else 0}
        elif op == "count":
            result = {"count": len(values)}
        else:
            return json.dumps({"error": f"Unknown operation '{operation}'. Use: sum, average, max, min, median, std, count, all", "status": "error"})

        return json.dumps({"result": result, "status": "success"})

    except json.JSONDecodeError:
        return json.dumps({"error": "Invalid JSON in data_string.", "status": "error"})
    except (ValueError, TypeError) as e:
        return json.dumps({"error": f"Data error: {str(e)}", "status": "error"})


# ============================================================
# TOOL 4: Report Formatter
# Returns structured report templates and metadata
# ============================================================

def format_report(report_type: str, data: str, period: str) -> str:
    """
    Returns a report structure template with metadata.
    report_type: sales, revenue, performance, quarterly, marketing
    data: JSON string with the data to be included
    period: e.g. 'Q1 2026', 'January 2026', 'FY2025'
    """
    try:
        report_type = report_type.lower().strip()

        templates = {
            "sales": {
                "header": f"{period} Sales Performance Report",
                "sections": ["Executive Summary", "Key Sales Metrics", "Monthly Breakdown", "Performance Analysis", "Recommendations"],
                "kpis": ["Total Revenue", "Average Monthly Revenue", "Peak Month", "Growth Rate", "Revenue per Period"]
            },
            "revenue": {
                "header": f"{period} Revenue Report",
                "sections": ["Executive Summary", "Revenue Breakdown", "Cost Analysis", "Profit Margins", "Forecast"],
                "kpis": ["Gross Revenue", "Net Revenue", "COGS", "Gross Margin", "YoY Growth"]
            },
            "performance": {
                "header": f"{period} Performance Report",
                "sections": ["Executive Summary", "KPI Dashboard", "Team Performance", "Goal Achievement", "Next Period Targets"],
                "kpis": ["Target Achievement %", "Average Performance Score", "Top Performer", "Areas for Improvement", "Trend"]
            },
            "quarterly": {
                "header": f"{period} Quarterly Business Report",
                "sections": ["Executive Overview", "Financial Highlights", "Operational Updates", "Market Position", "Strategic Next Steps"],
                "kpis": ["Total Revenue", "Operating Expenses", "Net Profit", "Customer Growth", "QoQ Growth Rate"]
            },
            "marketing": {
                "header": f"{period} Marketing Performance Report",
                "sections": ["Campaign Summary", "Lead Generation", "Conversion Metrics", "Channel Performance", "Budget Analysis"],
                "kpis": ["Total Leads", "Conversion Rate", "Cost per Lead", "ROI", "Top Channel"]
            }
        }

        template = templates.get(report_type, templates["sales"])

        return json.dumps({
            "header": template["header"],
            "sections": template["sections"],
            "kpis": template["kpis"],
            "timestamp": datetime.now().strftime("%B %d, %Y"),
            "period": period,
            "report_type": report_type,
            "data_received": data,
            "status": "success"
        })

    except Exception as e:
        return json.dumps({"error": f"Format error: {str(e)}", "status": "error"})


print("All 4 tool functions defined.")
print("Tools: Calculator | Web Search | Data Analyzer | Report Formatter")

---
## Step 5: Define Tool Schemas for Groq Function Calling

In [ ]:
# Tool schemas tell the model what tools are available and how to call them

calculator_tool = {
    "type": "function",
    "function": {
        "name": "calculate",
        "description": "Evaluate a mathematical or financial expression. Use for percentages, growth rates, totals, ratios, and any arithmetic. Example expressions: '50000 * 0.15', '(75000 - 50000) / 50000 * 100', 'sqrt(9)'.",
        "parameters": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "A valid Python math expression, e.g. '(75000 - 50000) / 50000 * 100' for growth rate."
                }
            },
            "required": ["expression"]
        }
    }
}

web_search_tool = {
    "type": "function",
    "function": {
        "name": "web_search",
        "description": "Search for business information, market trends, industry news, competitor analysis, and best practices. Use before writing emails or reports that require market context.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Search query, e.g. 'tech industry market trends 2026' or 'email best practices for sales'"
                }
            },
            "required": ["query"]
        }
    }
}

data_analyzer_tool = {
    "type": "function",
    "function": {
        "name": "analyze_data",
        "description": "Analyze business data: sales figures, revenue, KPIs, or any numeric dataset. Accepts JSON list or JSON object. Returns sum, average, max, min, median, std deviation, trend, and growth rate.",
        "parameters": {
            "type": "object",
            "properties": {
                "data_string": {
                    "type": "string",
                    "description": "JSON string of data. List: '[50000, 60000, 70000]'. Dict: '{\"Jan\": 50000, \"Feb\": 60000}'"
                },
                "operation": {
                    "type": "string",
                    "description": "Operation to perform: 'sum', 'average', 'max', 'min', 'median', 'std', 'count', or 'all'. Default: 'all'.",
                    "enum": ["sum", "average", "max", "min", "median", "std", "count", "all"]
                }
            },
            "required": ["data_string"]
        }
    }
}

report_formatter_tool = {
    "type": "function",
    "function": {
        "name": "format_report",
        "description": "Get a professional report structure and template. Returns section headers, KPI labels, and metadata. Use this before generating any business report to structure the output correctly.",
        "parameters": {
            "type": "object",
            "properties": {
                "report_type": {
                    "type": "string",
                    "description": "Type of report: 'sales', 'revenue', 'performance', 'quarterly', or 'marketing'",
                    "enum": ["sales", "revenue", "performance", "quarterly", "marketing"]
                },
                "data": {
                    "type": "string",
                    "description": "JSON string of the data to be included in the report."
                },
                "period": {
                    "type": "string",
                    "description": "Reporting period, e.g. 'Q1 2026', 'January 2026', 'FY2025'"
                }
            },
            "required": ["report_type", "data", "period"]
        }
    }
}

# Map tool names to Python functions
TOOL_FUNCTIONS = {
    "calculate":    calculate,
    "web_search":   web_search,
    "analyze_data": analyze_data,
    "format_report": format_report,
}

ALL_TOOLS = [calculator_tool, web_search_tool, data_analyzer_tool, report_formatter_tool]

print("Tool schemas defined.")
print(f"Total tools registered: {len(ALL_TOOLS)}")

---
## Step 6: Build the BusinessAssistant Class

In [ ]:
class BusinessAssistant:
    """
    Business Email and Report Manager
    Powered by Groq API (llama-3.3-70b-versatile)

    Features:
        1. Smart Email Writer         - Research + draft professional emails
        2. Report Generator           - Create structured business reports from data
        3. Meeting Summarizer         - Convert notes to structured summaries
        4. Data Analyzer              - Natural language queries on business data
        5. Client Communication       - Proposals, updates, follow-ups, responses

    Tools: Calculator, Web Search, Data Analyzer, Report Formatter
    """

    MODEL = "llama-3.3-70b-versatile"
    MAX_TOKENS = 1500

    def __init__(self, api_key: str):
        """Initialize the assistant with a Groq API key."""
        if not api_key or not api_key.strip():
            raise ValueError("GROQ_API_KEY is missing. Add it to Colab Secrets as 'groq'.")

        self.client = Groq(api_key=api_key)
        self.tools = ALL_TOOLS
        self.functions = TOOL_FUNCTIONS
        print("BusinessAssistant initialized.")
        print(f"Model: {self.MODEL}")

    # ----------------------------------------------------------
    # INTERNAL: Execute a tool call
    # ----------------------------------------------------------
    def _execute_tool(self, tool_name: str, tool_args: dict) -> str:
        """Execute a registered tool by name and return its JSON result."""
        if tool_name not in self.functions:
            return json.dumps({"error": f"Unknown tool: {tool_name}", "status": "error"})
        try:
            return self.functions[tool_name](**tool_args)
        except TypeError as e:
            return json.dumps({"error": f"Wrong arguments for '{tool_name}': {str(e)}", "status": "error"})
        except Exception as e:
            return json.dumps({"error": f"Tool '{tool_name}' failed: {str(e)}", "status": "error"})

    # ----------------------------------------------------------
    # INTERNAL: Call the LLM
    # ----------------------------------------------------------
    def _call_llm(self, messages: list, use_tools: bool = True, temperature: float = 0.6) -> object:
        """Call the Groq API. Returns the full response object."""
        kwargs = {
            "model": self.MODEL,
            "messages": messages,
            "max_tokens": self.MAX_TOKENS,
            "temperature": temperature,
        }
        if use_tools:
            kwargs["tools"] = self.tools
            kwargs["tool_choice"] = "auto"

        return self.client.chat.completions.create(**kwargs)

    # ----------------------------------------------------------
    # INTERNAL: Agentic tool-use loop
    # ----------------------------------------------------------
    def _run_agent_loop(self, messages: list, verbose: bool = True) -> str:
        """
        Keeps calling the LLM and executing tools until the model
        returns a final text response with no further tool calls.
        """
        iteration = 0
        max_iterations = 8  # Safety limit

        while iteration < max_iterations:
            iteration += 1
            response = self._call_llm(messages)
            msg = response.choices[0].message

            # No tool calls means we have the final answer
            if not msg.tool_calls:
                return msg.content or "(No response generated.)"

            # Append the assistant's tool-use message
            messages.append(msg)

            # Execute each tool call
            for tc in msg.tool_calls:
                tool_name = tc.function.name
                try:
                    tool_args = json.loads(tc.function.arguments)
                except json.JSONDecodeError:
                    tool_args = {}

                if verbose:
                    args_preview = ", ".join(f"{k}={repr(v)[:40]}" for k, v in tool_args.items())
                    print(f"  [Tool] {tool_name}({args_preview})")

                tool_result = self._execute_tool(tool_name, tool_args)

                messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": tool_result,
                })

        return "(Max iterations reached. Please try a simpler query.)"

    # ==========================================================
    # FEATURE 1: Smart Email Writer
    # ==========================================================
    def write_email(
        self,
        purpose: str,
        recipient: str = "stakeholder",
        tone: str = "formal",
        research_topic: str = None
    ) -> str:
        """
        Write a professional business email.

        Args:
            purpose       : What the email is about
            recipient     : 'client', 'team', 'stakeholder', or 'supplier'
            tone          : 'formal', 'professional', or 'friendly'
            research_topic: If provided, the assistant will search for market context first
        """
        # Input validation
        if not purpose or not purpose.strip():
            return "Error: Email purpose cannot be empty."

        recipient = recipient.lower().strip()
        valid_recipients = ["client", "team", "stakeholder", "supplier", "investor", "partner"]
        if recipient not in valid_recipients:
            print(f"  Warning: Unknown recipient type '{recipient}'. Defaulting to 'stakeholder'.")
            recipient = "stakeholder"

        tone = tone.lower().strip()
        valid_tones = ["formal", "professional", "friendly"]
        if tone not in valid_tones:
            print(f"  Warning: Unknown tone '{tone}'. Defaulting to 'professional'.")
            tone = "professional"

        system_prompt = f"""You are a senior business communication specialist. Write professional business emails.

Recipient type: {recipient.upper()}
Tone: {tone.upper()}

Tone guidelines:
- formal: Strictly professional. Full sentences. Titles and last names. No contractions.
- professional: Clear and business-appropriate. Balanced. Respectful but not stiff.
- friendly: Warm and approachable. Still professional. First names acceptable.

Output format (strictly follow this):
Subject: [subject line]

Dear [Recipient],

[Opening paragraph — establish purpose]

[Body paragraphs — details, data, context]

[Closing paragraph — call to action or next steps]

[Sign-off],
[Your Name]
[Title], [Company]"""

        research_instruction = ""
        if research_topic:
            research_instruction = f"Before writing, use web_search to research '{research_topic}' and incorporate relevant data or trends into the email."

        user_message = f"Write a {tone} {recipient} email for the following purpose:\n{purpose}\n{research_instruction}"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]

        print(f"\nFeature 1: Smart Email Writer")
        print(f"  Recipient: {recipient} | Tone: {tone} | Research: {'Yes' if research_topic else 'No'}")
        print("  Processing...")

        try:
            return self._run_agent_loop(messages)
        except Exception as e:
            return f"Error: Email generation failed — {str(e)}"

    # ==========================================================
    # FEATURE 2: Report Generator
    # ==========================================================
    def generate_report(
        self,
        report_type: str,
        data: dict or list,
        period: str
    ) -> str:
        """
        Generate a professional business report from data.

        Args:
            report_type : 'sales', 'revenue', 'performance', 'quarterly', 'marketing'
            data        : dict like {"Jan": 50000, "Feb": 65000} or list like [50000, 65000]
            period      : e.g. 'Q1 2026', 'January 2026'
        """
        # Input validation
        if not data:
            return "Error: Data cannot be empty."
        if not period or not period.strip():
            return "Error: Period cannot be empty."

        report_type = report_type.lower().strip()
        valid_types = ["sales", "revenue", "performance", "quarterly", "marketing"]
        if report_type not in valid_types:
            print(f"  Warning: Unknown report type '{report_type}'. Defaulting to 'sales'.")
            report_type = "sales"

        data_json = json.dumps(data)

        system_prompt = """You are a business analyst who creates professional reports.
When given data and a report template, produce a complete, well-structured business report.
Use the format_report tool to get the report structure, then use analyze_data to get metrics,
then use calculate for growth rates and percentages.
Write the final report in this structure:

============================================
[REPORT TITLE]
============================================
Date: [current date]
Period: [period]

EXECUTIVE SUMMARY
[2-3 sentences summarizing key findings]

KEY METRICS
[Bullet points with calculated values]

DATA ANALYSIS
[Detailed paragraph interpreting the numbers, trends, and what they mean for the business]

RECOMMENDATIONS
1. [Recommendation based on data]
2. [Recommendation based on data]
3. [Recommendation based on data]
4. [Recommendation based on data]
5. [Recommendation based on data]
============================================"""

        user_message = f"""Generate a {report_type} report for period: {period}

Data: {data_json}

Steps:
1. Call format_report to get the report structure
2. Call analyze_data with operation='all' to compute all metrics
3. Call calculate to compute growth rate: (last_value - first_value) / first_value * 100
4. Write the complete report using the structure and computed metrics"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]

        print(f"\nFeature 2: Report Generator")
        print(f"  Type: {report_type} | Period: {period} | Data points: {len(data) if hasattr(data, '__len__') else 'N/A'}")
        print("  Processing...")

        try:
            return self._run_agent_loop(messages)
        except Exception as e:
            return f"Error: Report generation failed — {str(e)}"

    # ==========================================================
    # FEATURE 3: Meeting Summarizer
    # ==========================================================
    def summarize_meeting(
        self,
        notes: str,
        date: str = None,
        attendees: list = None
    ) -> str:
        """
        Convert raw meeting notes into a structured summary with action items.

        Args:
            notes     : Raw meeting notes as a string
            date      : Meeting date (defaults to today if not provided)
            attendees : Optional list of attendees
        """
        # Input validation
        if not notes or not notes.strip():
            return "Error: Meeting notes cannot be empty."
        if len(notes.split()) < 10:
            return "Error: Meeting notes are too short. Please provide more detail."

        if not date:
            date = datetime.now().strftime("%B %d, %Y")

        attendees_str = ", ".join(attendees) if attendees else "Not specified"

        system_prompt = """You are an executive assistant who produces structured meeting summaries.
Extract all important information from meeting notes and organize it clearly.

ALWAYS produce output in EXACTLY this format:

===========================================
MEETING SUMMARY
===========================================
Date: [date]
Attendees: [attendees]

SUMMARY
[2-3 sentence overview of what the meeting accomplished]

KEY DISCUSSION POINTS
- [point 1]
- [point 2]
- [point 3]
(add as many as needed)

DECISIONS MADE
1. [decision 1]
2. [decision 2]
(add as many as needed)

ACTION ITEMS
- [Owner / Team]: [Action to be taken] by [deadline if mentioned]
(add as many as needed, write 'No specific assignments mentioned' if none)

NEXT MEETING: [date/time if mentioned, otherwise 'Not scheduled']
==========================================="""

        user_message = f"""Summarize these meeting notes.

Date: {date}
Attendees: {attendees_str}

Raw Notes:
{notes}"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]

        print(f"\nFeature 3: Meeting Summarizer")
        print(f"  Date: {date} | Attendees: {attendees_str}")
        print("  Processing...")

        try:
            # Meeting summarizer does not need tools
            response = self._call_llm(messages, use_tools=False)
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error: Meeting summarization failed — {str(e)}"

    # ==========================================================
    # FEATURE 4: Data Analyzer
    # ==========================================================
    def analyze_business_data(
        self,
        query: str,
        data: dict or list
    ) -> str:
        """
        Answer natural language queries about business data.

        Args:
            query : Natural language question, e.g. 'What is the average monthly revenue?'
            data  : Business data as dict or list
        """
        # Input validation
        if not query or not query.strip():
            return "Error: Query cannot be empty."
        if not data:
            return "Error: Data cannot be empty."

        data_json = json.dumps(data)

        system_prompt = """You are a business data analyst. Answer natural language queries about business data.

Use analyze_data and calculate tools to compute precise values.
Always produce output in this format:

ANALYSIS RESULTS
==================
[Answer to the specific query with computed values]

KEY METRICS
[Bullet list of all relevant computed metrics with proper formatting]

INTERPRETATION
[2-3 sentences explaining what these numbers mean for the business]

RECOMMENDATION
[1-2 actionable recommendations based on the data]"""

        user_message = f"""Answer this business query using the provided data.

Query: {query}

Data: {data_json}

Use analyze_data (operation='all') first to compute all statistics, then use calculate for any additional metrics needed."""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]

        print(f"\nFeature 4: Business Data Analyzer")
        print(f"  Query: {query[:80]}{'...' if len(query) > 80 else ''}")
        print("  Processing...")

        try:
            return self._run_agent_loop(messages)
        except Exception as e:
            return f"Error: Data analysis failed — {str(e)}"

    # ==========================================================
    # FEATURE 5: Client Communication Drafter
    # ==========================================================
    def draft_client_communication(
        self,
        comm_type: str,
        client: str,
        context: str,
        tone: str = "professional"
    ) -> str:
        """
        Draft professional client communications.

        Args:
            comm_type : 'proposal', 'status_update', 'response', 'follow_up'
            client    : Client name or company
            context   : Details about the communication
            tone      : 'formal', 'professional', or 'friendly'
        """
        # Input validation
        if not context or not context.strip():
            return "Error: Context cannot be empty."
        if not client or not client.strip():
            return "Error: Client name cannot be empty."

        comm_type = comm_type.lower().strip().replace(" ", "_")
        valid_types = ["proposal", "status_update", "response", "follow_up"]
        if comm_type not in valid_types:
            print(f"  Warning: Unknown communication type '{comm_type}'. Defaulting to 'response'.")
            comm_type = "response"

        tone = tone.lower().strip()
        if tone not in ["formal", "professional", "friendly"]:
            tone = "professional"

        type_guidance = {
            "proposal": "Write a compelling project or business proposal. Include scope, timeline, value proposition, and call to action.",
            "status_update": "Write a clear project status update. Include progress made, current status, upcoming milestones, and any blockers.",
            "response": "Write a professional response to a client inquiry. Be clear, helpful, and address all points raised.",
            "follow_up": "Write a polite but purposeful follow-up message. Reference previous communication and include a clear next step."
        }

        system_prompt = f"""You are a senior client relationship manager. Draft professional client communications.

Communication type: {comm_type.upper().replace('_', ' ')}
Tone: {tone.upper()}
Instruction: {type_guidance[comm_type]}

Output a complete, ready-to-send communication with:
- Professional greeting using client name
- Well-structured body (2-4 paragraphs appropriate to the type)
- Clear call-to-action or next step
- Professional closing with signature placeholder"""

        user_message = f"""Draft a {tone} {comm_type.replace('_', ' ')} for client: {client}

Context and details:
{context}"""

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]

        print(f"\nFeature 5: Client Communication Drafter")
        print(f"  Type: {comm_type} | Client: {client} | Tone: {tone}")
        print("  Processing...")

        try:
            # Client comms rarely need tools; disable to reduce latency
            response = self._call_llm(messages, use_tools=False)
            return response.choices[0].message.content.strip()
        except Exception as e:
            return f"Error: Client communication drafting failed — {str(e)}"

    # ==========================================================
    # MAIN ROUTER: process_request
    # ==========================================================
    def process_request(self, request: str) -> str:
        """
        Auto-routes a natural language request to the appropriate feature.
        Handles mixed requests by selecting the best-matching capability.
        """
        if not request or not request.strip():
            return "Error: Request cannot be empty."

        r = request.lower()

        # Route: Email Writer
        email_keywords = ["write email", "draft email", "compose email", "send email", "email to"]
        if any(kw in r for kw in email_keywords):
            tone = "professional"
            if "formal" in r:   tone = "formal"
            if "friendly" in r: tone = "friendly"
            recipient = "stakeholder"
            for rec in ["client", "team", "supplier", "investor", "partner"]:
                if rec in r:
                    recipient = rec
                    break
            return self.write_email(request, recipient=recipient, tone=tone)

        # Route: Report Generator
        report_keywords = ["generate report", "create report", "write report", "business report", "quarterly report", "sales report"]
        if any(kw in r for kw in report_keywords):
            return "To generate a report, call assistant.generate_report(report_type, data, period) with your data."

        # Route: Meeting Summarizer
        meeting_keywords = ["meeting notes", "summarize meeting", "meeting summary", "action items from"]
        if any(kw in r for kw in meeting_keywords):
            return "To summarize a meeting, call assistant.summarize_meeting(notes, date, attendees) with your notes."

        # Route: Data Analysis
        data_keywords = ["analyze data", "analyze sales", "average revenue", "total sales", "growth rate", "what is the"]
        if any(kw in r for kw in data_keywords):
            return "To analyze data, call assistant.analyze_business_data(query, data) with your dataset."

        # Route: Client Communication
        client_keywords = ["proposal", "status update", "follow up", "follow-up", "client response", "draft for client"]
        if any(kw in r for kw in client_keywords):
            return "To draft client communication, call assistant.draft_client_communication(comm_type, client, context, tone)."

        # Fallback: general assistant response
        messages = [
            {"role": "system", "content": "You are a helpful business assistant. Answer business questions clearly and concisely. Use tools if needed."},
            {"role": "user",   "content": request}
        ]
        try:
            return self._run_agent_loop(messages)
        except Exception as e:
            return f"Error processing request: {str(e)}"


print("BusinessAssistant class defined successfully.")

---
## Step 7: Initialize the Assistant

In [ ]:
assistant = BusinessAssistant(api_key=GROQ_API_KEY)

print("""
============================================================
  Business Email and Report Manager
  Powered by Groq API (llama-3.3-70b-versatile)
============================================================
  Features:
    1. Smart Email Writer
    2. Report Generator
    3. Meeting Summarizer
    4. Business Data Analyzer
    5. Client Communication Drafter

  Tools: Calculator | Web Search | Data Analyzer | Report Formatter
============================================================
""")

---
## Step 8: Demo All Features

Run each cell below to test every feature and tool.

### Demo 1 — Smart Email Writer (Formal, no research)

In [ ]:
result = assistant.write_email(
    purpose="Announce Q2 2026 sales results to stakeholders. Total revenue was PKR 12.5 million, up 28% from Q1.",
    recipient="stakeholder",
    tone="formal"
)
print(result)

### Demo 2 — Smart Email Writer (Friendly, with market research)

In [ ]:
result = assistant.write_email(
    purpose="Introduce our new AI-powered inventory management software to potential clients",
    recipient="client",
    tone="friendly",
    research_topic="AI adoption in business 2026"   # Triggers web_search tool
)
print(result)

### Demo 3 — Report Generator (Quarterly Sales Report)

In [ ]:
quarterly_sales = {
    "January":  50000,
    "February": 58000,
    "March":    65000,
    "April":    61000,
    "May":      72000,
    "June":     80000
}

result = assistant.generate_report(
    report_type="quarterly",
    data=quarterly_sales,
    period="H1 2026"
)
print(result)

### Demo 4 — Meeting Summarizer

In [ ]:
raw_notes = """
We started with a review of Q2 marketing results. Campaigns performed below expectations — 
leads were down 15% compared to Q1. Ali presented root cause analysis: budget allocation 
was suboptimal and LinkedIn ads were paused mid-quarter due to creative delays.

Team agreed to increase Q3 marketing budget from Rs 400k to Rs 550k. Fatima will lead 
the Q3 campaign strategy. She needs to submit a campaign brief by May 30th.

Usman from design team to deliver 3 ad creatives per channel per month. HR to post job 
listing for a content writer by end of this week.

Decision was made to stop Google Display Ads and shift full budget to LinkedIn and 
email marketing based on conversion data.

Next meeting scheduled for June 5th, 10am.
"""

result = assistant.summarize_meeting(
    notes=raw_notes,
    date="May 22, 2026",
    attendees=["Ali (Marketing)", "Fatima (Campaign Lead)", "Usman (Design)", "HR Team"]
)
print(result)

### Demo 5 — Business Data Analyzer

In [ ]:
monthly_revenue = {
    "Jan": 50000,
    "Feb": 55000,
    "Mar": 62000,
    "Apr": 59000,
    "May": 68000,
    "Jun": 75000
}

result = assistant.analyze_business_data(
    query="What is the average monthly revenue, overall growth rate, and which month performed best?",
    data=monthly_revenue
)
print(result)

In [ ]:
# Second data analysis example with a list
conversion_rates = [2.1, 2.4, 2.2, 2.8, 3.1, 3.4, 3.0, 3.6]

result = assistant.analyze_business_data(
    query="Analyze these monthly conversion rates and tell me if we are improving. What is the trend?",
    data=conversion_rates
)
print(result)

### Demo 6 — Client Communication Drafter (Proposal)

In [ ]:
result = assistant.draft_client_communication(
    comm_type="proposal",
    client="NexaTech Solutions",
    context="""
    Project: Complete ERP system implementation
    Timeline: 4 months
    Budget: PKR 2.5 million
    Scope: Accounting, HR, Inventory, and Reporting modules
    Our team: 3 developers, 1 project manager, 1 QA engineer
    Previous work: Successfully delivered similar project for a retail chain in 2025
    """,
    tone="professional"
)
print(result)

### Demo 7 — Client Communication Drafter (Status Update)

In [ ]:
result = assistant.draft_client_communication(
    comm_type="status_update",
    client="Al-Noor Enterprises",
    context="""
    Project: Website redesign (started April 1, 2026)
    Current status: 60% complete, on schedule
    Completed: Homepage, About, Services pages and mobile responsiveness
    Remaining: Contact page, blog section, SEO optimization, testing
    Expected delivery: June 15, 2026
    Issue to flag: Client logo files received in low resolution, need high-res version
    """,
    tone="professional"
)
print(result)

### Demo 8 — Tool Unit Tests

In [ ]:
print("Running Tool Unit Tests")
print("=" * 55)

tests = [
    # (tool_function, kwargs, description)
    (calculate,    {"expression": "50000 * 0.15"},                                                         "Calculator: 15% of 50000"),
    (calculate,    {"expression": "(80000 - 50000) / 50000 * 100"},                                       "Calculator: Growth rate 50k to 80k"),
    (calculate,    {"expression": "(72000 + 68000 + 80000) / 3"},                                         "Calculator: 3-month average"),
    (web_search,   {"query": "market trends 2026"},                                                        "Web Search: market trends"),
    (web_search,   {"query": "sales strategy best practices"},                                             "Web Search: sales strategy"),
    (analyze_data, {"data_string": '[50000, 58000, 65000, 72000, 80000]', "operation": "all"},             "Data Analyzer: full analysis"),
    (analyze_data, {"data_string": '{"Jan": 50000, "Feb": 65000, "Mar": 80000}', "operation": "average"}, "Data Analyzer: dict average"),
    (format_report, {"report_type": "quarterly", "data": '[50000, 65000, 80000]', "period": "Q1 2026"},   "Report Formatter: quarterly"),
    (format_report, {"report_type": "sales", "data": '{"Jan": 50000}', "period": "January 2026"},         "Report Formatter: sales"),
]

passed = 0
failed = 0

for func, kwargs, description in tests:
    try:
        result = func(**kwargs)
        parsed = json.loads(result)
        if parsed.get("status") == "error":
            print(f"  FAIL  [{description}]: {parsed.get('error')}")
            failed += 1
        else:
            print(f"  PASS  [{description}]")
            passed += 1
    except json.JSONDecodeError as e:
        print(f"  ERROR [{description}]: JSON parse failed — {e}")
        failed += 1
    except Exception as e:
        print(f"  ERROR [{description}]: {e}")
        failed += 1

print("=" * 55)
print(f"Results: {passed} passed, {failed} failed out of {len(tests)} tests")

---
## Step 9: Interactive Mode

Run this cell for a menu-driven interactive session.

In [ ]:
def interactive_session(assistant):
    """Menu-driven interactive session for the BusinessAssistant."""

    menu = """
============================================================
  BUSINESS ASSISTANT — INTERACTIVE MENU
============================================================
  1. Write Email
  2. Generate Report
  3. Summarize Meeting
  4. Analyze Business Data
  5. Draft Client Communication
  0. Exit
============================================================"""

    print(menu)

    while True:
        try:
            choice = input("\nSelect option (0-5): ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if choice == "0":
            print("Session ended. Goodbye.")
            break

        elif choice == "1":
            print("\n--- Email Writer ---")
            purpose   = input("Email purpose: ").strip()
            recipient = input("Recipient type (client/team/stakeholder/supplier) [default: stakeholder]: ").strip() or "stakeholder"
            tone      = input("Tone (formal/professional/friendly) [default: formal]: ").strip() or "formal"
            research  = input("Research topic (press Enter to skip): ").strip() or None
            if purpose:
                result = assistant.write_email(purpose, recipient=recipient, tone=tone, research_topic=research)
                print("\n" + result)
            else:
                print("Purpose cannot be empty.")

        elif choice == "2":
            print("\n--- Report Generator ---")
            report_type = input("Report type (sales/revenue/performance/quarterly/marketing): ").strip() or "sales"
            period      = input("Period (e.g. Q1 2026): ").strip() or "Q1 2026"
            print("Enter data as JSON (e.g. {\"Jan\": 50000, \"Feb\": 60000} or [50000, 60000]):")
            data_str = input("Data: ").strip()
            try:
                data = json.loads(data_str)
                result = assistant.generate_report(report_type, data, period)
                print("\n" + result)
            except json.JSONDecodeError:
                print("Invalid JSON. Please enter valid JSON data.")

        elif choice == "3":
            print("\n--- Meeting Summarizer ---")
            print("Paste meeting notes (type END on a new line when done):")
            lines = []
            while True:
                line = input()
                if line.strip().upper() == "END":
                    break
                lines.append(line)
            notes = "\n".join(lines)
            date  = input("Meeting date [default: today]: ").strip() or None
            att_str = input("Attendees, comma-separated (press Enter to skip): ").strip()
            attendees = [a.strip() for a in att_str.split(",")] if att_str else None
            if notes:
                result = assistant.summarize_meeting(notes, date=date, attendees=attendees)
                print("\n" + result)
            else:
                print("Notes cannot be empty.")

        elif choice == "4":
            print("\n--- Business Data Analyzer ---")
            query = input("Your question about the data: ").strip()
            print("Enter data as JSON (e.g. {\"Jan\": 50000, \"Feb\": 60000} or [50000, 60000]):")
            data_str = input("Data: ").strip()
            try:
                data = json.loads(data_str)
                result = assistant.analyze_business_data(query, data)
                print("\n" + result)
            except json.JSONDecodeError:
                print("Invalid JSON. Please enter valid JSON data.")

        elif choice == "5":
            print("\n--- Client Communication Drafter ---")
            comm_type = input("Type (proposal/status_update/response/follow_up): ").strip() or "response"
            client    = input("Client name/company: ").strip()
            context   = input("Context and details: ").strip()
            tone      = input("Tone (formal/professional/friendly) [default: professional]: ").strip() or "professional"
            if client and context:
                result = assistant.draft_client_communication(comm_type, client, context, tone)
                print("\n" + result)
            else:
                print("Client name and context are required.")

        else:
            print("Invalid option. Please select 0-5.")

# Run the interactive session
interactive_session(assistant)

---
## Step 10: Generate Submission Files

In [ ]:
import os

# Create examples directory
os.makedirs("examples", exist_ok=True)

# ── requirements.txt ──────────────────────────────────────────
with open("requirements.txt", "w") as f:
    f.write("groq>=0.9.0\npython-dotenv>=1.0.0\n")
print("Created: requirements.txt")

# ── .env.example ───────────────────────────────────────────────
with open(".env.example", "w") as f:
    f.write("# Copy this file to .env and fill in your key\nGROQ_API_KEY=your-groq-api-key-here\n")
print("Created: .env.example")

# ── examples/email_examples.txt ───────────────────────────────
email_examples = """EMAIL WRITER EXAMPLES
======================

Example 1 — Formal stakeholder email:
  assistant.write_email(
      purpose="Announce Q2 sales results, total revenue PKR 12.5M, up 28% from Q1",
      recipient="stakeholder",
      tone="formal"
  )

Example 2 — Friendly client email with research:
  assistant.write_email(
      purpose="Introduce our new AI inventory software",
      recipient="client",
      tone="friendly",
      research_topic="AI adoption in business 2026"
  )

Example 3 — Professional team email:
  assistant.write_email(
      purpose="Announce new remote work policy effective June 1, 2026",
      recipient="team",
      tone="professional"
  )
"""
with open("examples/email_examples.txt", "w") as f:
    f.write(email_examples)
print("Created: examples/email_examples.txt")

# ── examples/report_examples.txt ──────────────────────────────
report_examples = """REPORT GENERATOR EXAMPLES
==========================

Example 1 — Quarterly sales report:
  data = {"Jan": 50000, "Feb": 58000, "Mar": 65000}
  assistant.generate_report("quarterly", data, "Q1 2026")

Example 2 — Revenue report with list data:
  data = [120000, 135000, 142000, 150000]
  assistant.generate_report("revenue", data, "H1 2026")

Example 3 — Marketing performance report:
  data = {"Jan": 450, "Feb": 520, "Mar": 610, "Apr": 580}
  assistant.generate_report("marketing", data, "Q1 2026")
"""
with open("examples/report_examples.txt", "w") as f:
    f.write(report_examples)
print("Created: examples/report_examples.txt")

# ── examples/meeting_examples.txt ─────────────────────────────
meeting_examples = """MEETING SUMMARIZER EXAMPLES
============================

Example 1:
  notes = '''
  Discussed Q3 product roadmap. Decided to prioritize mobile app over web dashboard.
  Ahmed to complete UI mockups by May 30. Sana will coordinate with dev team.
  Budget for Q3 approved at PKR 800k. Next meeting June 1.
  '''
  assistant.summarize_meeting(
      notes=notes,
      date="May 20, 2026",
      attendees=["Ahmed (Design)", "Sana (PM)", "Dev Team Lead"]
  )
"""
with open("examples/meeting_examples.txt", "w") as f:
    f.write(meeting_examples)
print("Created: examples/meeting_examples.txt")

# ── README.md ──────────────────────────────────────────────────
readme = """# Business Email and Report Manager
**Agentic AI Bootcamp - atomcamp | Weekly Project**

A complete AI-powered business communication and reporting assistant built with Groq API (free).

---

## Features

| # | Feature | Description |
|---|---------|-------------|
| 1 | Smart Email Writer | Writes professional emails in any tone; can research topics before drafting |
| 2 | Report Generator | Creates structured business reports from data with computed metrics |
| 3 | Meeting Summarizer | Converts raw notes to structured summaries with action items |
| 4 | Business Data Analyzer | Answers natural language queries about sales and financial data |
| 5 | Client Communication Drafter | Creates proposals, status updates, follow-ups, and responses |

## Tools

| # | Tool | Description |
|---|------|-------------|
| 1 | Calculator | Evaluates financial expressions — percentages, growth rates, totals |
| 2 | Web Search | Mock search returning realistic business and market data |
| 3 | Data Analyzer | Computes sum, average, max, min, median, std, trend for datasets |
| 4 | Report Formatter | Returns structured report templates with section headers and KPIs |

---

## Installation

```bash
pip install -r requirements.txt
```

Copy `.env.example` to `.env` and add your Groq API key:
```
GROQ_API_KEY=your-key-here
```

## Usage

### Google Colab:
1. Add your Groq API key to Colab Secrets (lock icon) as `groq`
2. Run all cells in `Business_Email_Report_Manager.ipynb`

### Direct API calls:
```python
from business_assistant import BusinessAssistant
import os

assistant = BusinessAssistant(api_key=os.environ["GROQ_API_KEY"])

# Write an email
print(assistant.write_email("Announce new product launch", recipient="client", tone="formal"))

# Generate a report
data = {"Jan": 50000, "Feb": 65000, "Mar": 80000}
print(assistant.generate_report("sales", data, "Q1 2026"))

# Summarize meeting
print(assistant.summarize_meeting(notes="...", date="May 22, 2026"))

# Analyze data
print(assistant.analyze_business_data("What is the average and growth rate?", data))

# Draft client communication
print(assistant.draft_client_communication("proposal", "ABC Corp", "Website redesign, 3 months, $50k"))
```

## File Structure

```
Business_Email_Report_Manager.ipynb  -- Main notebook
requirements.txt                     -- Python dependencies
.env.example                         -- API key template
README.md                            -- This file
examples/
  email_examples.txt
  report_examples.txt
  meeting_examples.txt
```

## Getting a Free Groq API Key

1. Go to https://console.groq.com
2. Sign up for a free account
3. Navigate to API Keys and create a new key
4. In Google Colab: click the lock icon (Secrets) and add key name `groq`

---
*Built for atomcamp Agentic AI Bootcamp — Session 2 Weekly Project*
"""
with open("README.md", "w") as f:
    f.write(readme)
print("Created: README.md")

print("\nAll submission files generated successfully.")

---
## Submission Checklist

| Requirement | Status |
|-------------|--------|
| Feature 1: Smart Email Writer (formal/friendly/professional, with research) | Done |
| Feature 2: Report Generator (executive summary, metrics, recommendations) | Done |
| Feature 3: Meeting Summarizer (key points, decisions, action items) | Done |
| Feature 4: Business Data Analyzer (natural language queries) | Done |
| Feature 5: Client Communication Drafter (proposal/update/response/follow-up) | Done |
| Tool 1: Calculator | Done |
| Tool 2: Web Search | Done |
| Tool 3: Data Analyzer | Done |
| Tool 4: Report Formatter | Done |
| Agentic loop (multi-tool calls per request) | Done |
| Input validation on all features | Done |
| Error handling on all tool calls and LLM calls | Done |
| BusinessAssistant class with process_request router | Done |
| README.md | Done |
| requirements.txt | Done |
| .env.example | Done |
| examples/ directory | Done |
| Uses free Groq API | Done |